# Xception Hyperparameter Tuning

This is a true tuned version of the baseline notebook. It tests several hyperparameter combinations, chooses the best configuration using a validation subset from the training split, then retrains/evaluates the best configuration on the official CIFAKE test split. The official test split is not used for choosing hyperparameters.

In [ ]:
from pathlib import Path
import copy
import os
import importlib.util
import itertools
import json
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from PIL import Image, ImageFile
from tqdm.auto import tqdm

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
pin_memory = device.type == "cuda"
num_workers = 0
ImageFile.LOAD_TRUNCATED_IMAGES = True
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA GPU was not detected. The notebook will run on CPU unless you install/use a CUDA-enabled PyTorch environment.")

import subprocess
import sys
try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])
    import timm
IN_COLAB = os.getenv("COLAB_RELEASE_TAG") is not None or (importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None)


In [ ]:
# Mount Google Drive in Colab
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/gdrive", force_remount=True)


In [ ]:
def has_cifake_splits(data_dir):
    data_dir = Path(data_dir).expanduser().resolve()
    return (data_dir / "train").exists() and (data_dir / "test").exists()


def find_project_root():
    # In Colab, keep the dataset on local runtime storage. Extracting many small
    # CIFAKE files into Google Drive can appear frozen for a very long time.
    if IN_COLAB:
        return Path("/content/TDK_AI_Detection")

    cwd = Path.cwd().resolve()
    candidates = []

    env_project_root = os.getenv("TDK_PROJECT_ROOT")
    if env_project_root:
        candidates.append(Path(env_project_root).expanduser())

    env_data_root = os.getenv("CIFAKE_ROOT")
    if env_data_root:
        data_root = Path(env_data_root).expanduser()
        if has_cifake_splits(data_root):
            return data_root.resolve().parent
        candidates.append(data_root)

    candidates.extend([cwd, *list(cwd.parents)[:4]])
    candidates.extend([
        Path("/content/TDK_AI_Detection"),
        Path("/content/gdrive/MyDrive/TDK_AI_Detection"),
        Path("/content/drive/MyDrive/TDK_AI_Detection"),
    ])

    seen = set()
    for candidate in candidates:
        candidate = Path(candidate).expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_cifake_splits(candidate / "CIFAKE_FULL"):
            return candidate

    # Local fallback: if the notebook is opened from the Tuned Models folder,
    # save outputs/models beside that folder and put CIFAKE_FULL at the project root.
    if cwd.name.lower() == "tuned models":
        return cwd.parent
    return cwd

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "CIFAKE_FULL"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
MODEL_DIR = PROJECT_ROOT / "models" / "tuned"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tuned"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)

def ensure_cifake_dataset():
    expected_dirs = [
        TRAIN_DIR / "FAKE",
        TRAIN_DIR / "REAL",
        TEST_DIR / "FAKE",
        TEST_DIR / "REAL",
    ]
    if all(folder.exists() for folder in expected_dirs):
        print("CIFAKE_FULL already exists:", DATA_DIR)
        return

    import os
    import shutil
    import subprocess
    import sys
    import zipfile

    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    kaggle_dir = Path("/root/.kaggle") if IN_COLAB else Path(os.getenv("KAGGLE_CONFIG_DIR", Path.home() / ".kaggle"))
    kaggle_json = kaggle_dir / "kaggle.json"

    def install_kaggle_json():
        if kaggle_json.exists():
            print("Using existing Kaggle credentials:", kaggle_json)
            return

        candidates = [
            Path("/content/kaggle.json"),
            Path("/content/conncet/kaggle.json"),
            Path("/content/connect/kaggle.json"),
            Path.cwd() / "kaggle.json",
            Path.cwd() / "conncet" / "kaggle.json",
            Path.cwd() / "connect" / "kaggle.json",
            PROJECT_ROOT / "kaggle.json",
            PROJECT_ROOT / "conncet" / "kaggle.json",
            PROJECT_ROOT / "connect" / "kaggle.json",
            Path("/content/gdrive/MyDrive/kaggle.json"),
            Path("/content/gdrive/MyDrive/conncet/kaggle.json"),
            Path("/content/gdrive/MyDrive/connect/kaggle.json"),
            Path("/content/drive/MyDrive/kaggle.json"),
            Path("/content/drive/MyDrive/conncet/kaggle.json"),
            Path("/content/drive/MyDrive/connect/kaggle.json"),
        ]
        for candidate in candidates:
            if candidate.exists():
                kaggle_dir.mkdir(parents=True, exist_ok=True)
                shutil.copy2(candidate, kaggle_json)
                os.chmod(kaggle_json, 0o600)
                print("Installed Kaggle credentials from:", candidate)
                return

        raise FileNotFoundError(
            "kaggle.json was not found. In Colab, upload it to /content/kaggle.json, "
            "/content/conncet/kaggle.json, /content/connect/kaggle.json, or put it in "
            f"{PROJECT_ROOT / 'kaggle.json'}."
        )

    install_kaggle_json()

    try:
        import kaggle  # noqa: F401
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])

    dataset_slug = "birdy654/cifake-real-and-ai-generated-synthetic-images"
    zip_path = PROJECT_ROOT / "cifake-real-and-ai-generated-synthetic-images.zip"

    if not zip_path.exists():
        print("Downloading CIFAKE from Kaggle into:", PROJECT_ROOT)
        kaggle_cmd = shutil.which("kaggle")
        command = ([kaggle_cmd] if kaggle_cmd else [sys.executable, "-m", "kaggle"]) + [
            "datasets",
            "download",
            "-d",
            dataset_slug,
            "-p",
            str(PROJECT_ROOT),
        ]
        result = subprocess.run(command, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        if result.returncode != 0:
            raise RuntimeError(
                "Kaggle download failed. Check the message above. Common causes: invalid kaggle.json, "
                "Kaggle account not verified, or the dataset/API is unavailable."
            )

    print("Extracting CIFAKE into:", DATA_DIR)
    with zipfile.ZipFile(zip_path, "r") as zip_file:
        members = zip_file.infolist()
        for member in tqdm(members, desc="Extracting CIFAKE", unit="file"):
            zip_file.extract(member, DATA_DIR)

    missing = [folder for folder in expected_dirs if not folder.exists()]
    if missing:
        raise FileNotFoundError(f"CIFAKE extraction finished, but expected folders are missing: {missing}")


ensure_cifake_dataset()

if not TRAIN_DIR.exists() or not TEST_DIR.exists():
    raise FileNotFoundError("Expected CIFAKE_FULL/train and CIFAKE_FULL/test under the project root.")

class_names = ["fake", "real"]
normalize_mean = [0.485, 0.456, 0.406]
normalize_std = [0.229, 0.224, 0.225]


def make_transforms(image_size, augment):
    if augment:
        return transforms.Compose([
            transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=normalize_mean, std=normalize_std),
        ])
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=normalize_mean, std=normalize_std),
    ])


def is_valid_image_file(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


def normalized_imagefolder(split_dir, image_size, augment):
    raw_unfiltered = datasets.ImageFolder(split_dir)
    raw = datasets.ImageFolder(
        split_dir,
        transform=make_transforms(image_size, augment),
        is_valid_file=is_valid_image_file,
    )
    skipped = len(raw_unfiltered.samples) - len(raw.samples)
    if skipped:
        print(f"Skipped {skipped} unreadable image(s) in {split_dir}.")
    lower_to_original = {name.lower(): idx for name, idx in raw.class_to_idx.items()}
    fake_idx = lower_to_original["fake"]
    real_idx = lower_to_original["real"]
    remap = {fake_idx: 0, real_idx: 1}
    raw.target_transform = lambda target: remap[target]
    normalized_targets = np.array([remap[target] for _, target in raw.samples])
    return raw, normalized_targets


def balanced_indices(targets, per_class, rng):
    selected = []
    for label in [0, 1]:
        label_idx = np.where(targets == label)[0]
        if len(label_idx) < per_class:
            raise ValueError(f"Not enough samples for class {label}: requested {per_class}, found {len(label_idx)}")
        selected.extend(rng.choice(label_idx, size=per_class, replace=False).tolist())
    rng.shuffle(selected)
    return selected


def make_tuning_splits(image_size, train_per_class=2000, val_per_class=500):
    dataset, targets = normalized_imagefolder(TRAIN_DIR, image_size, augment=False)
    rng = np.random.default_rng(seed)
    train_idx = []
    val_idx = []
    for label in [0, 1]:
        label_idx = np.where(targets == label)[0]
        chosen = rng.choice(label_idx, size=train_per_class + val_per_class, replace=False)
        train_idx.extend(chosen[:train_per_class].tolist())
        val_idx.extend(chosen[train_per_class:].tolist())
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx


def metrics_from_predictions(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_fake_positive": float(precision_score(y_true, y_pred, pos_label=0, zero_division=0)),
        "recall_fake_positive": float(recall_score(y_true, y_pred, pos_label=0, zero_division=0)),
        "f1_fake_positive": float(f1_score(y_true, y_pred, pos_label=0, zero_division=0)),
        "precision_real_positive": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "recall_real_positive": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "f1_real_positive": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "weighted_precision": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "weighted_recall": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


In [ ]:
def build_model(hp):
    model = timm.create_model("xception", pretrained=True, num_classes=2)
    if hp["freeze_backbone"]:
        for name, param in model.named_parameters():
            if "fc" not in name and "classifier" not in name and "head" not in name:
                param.requires_grad = False
    return model

In [ ]:
MODEL_NAME = "xception_hyperparameter_tuned"
TUNING_TRAIN_PER_CLASS = 2000
TUNING_VAL_PER_CLASS = 500
FINAL_EPOCHS = 10

HYPERPARAMETER_GRID = [
    {"image_size": 224, "batch_size": 16, "learning_rate": 1e-3, "weight_decay": 0.0, "optimizer": "AdamW", "augment": True, "freeze_backbone": True, "trial_epochs": 3},
    {"image_size": 224, "batch_size": 16, "learning_rate": 1e-4, "weight_decay": 1e-4, "optimizer": "AdamW", "augment": True, "freeze_backbone": True, "trial_epochs": 3},
    {"image_size": 224, "batch_size": 16, "learning_rate": 1e-4, "weight_decay": 1e-4, "optimizer": "AdamW", "augment": True, "freeze_backbone": False, "trial_epochs": 3},
    {"image_size": 224, "batch_size": 16, "learning_rate": 1e-5, "weight_decay": 1e-4, "optimizer": "AdamW", "augment": True, "freeze_backbone": False, "trial_epochs": 3},
]

TUNING_RESULTS_PATH = OUTPUT_DIR / f"{MODEL_NAME}_tuning_results.json"
BEST_CONFIG_PATH = OUTPUT_DIR / f"{MODEL_NAME}_best_config.json"
FINAL_RESULTS_PATH = OUTPUT_DIR / f"{MODEL_NAME}_final_results.json"
BEST_MODEL_PATH = MODEL_DIR / f"{MODEL_NAME}_best.pth"


## Hyperparameter Tuning Loop

The grid below is the evidence that this notebook is tuned: it runs multiple values for learning rate, batch size, optimizer/weight decay, augmentation, and frozen/unfrozen backbone choices where appropriate.

In [ ]:
def make_optimizer(model, hp):
    params = [p for p in model.parameters() if p.requires_grad]
    if hp["optimizer"] == "Adam":
        return optim.Adam(params, lr=hp["learning_rate"], weight_decay=hp["weight_decay"])
    if hp["optimizer"] == "AdamW":
        return optim.AdamW(params, lr=hp["learning_rate"], weight_decay=hp["weight_decay"])
    raise ValueError(f"Unsupported optimizer: {hp['optimizer']}")


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    seen = 0
    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        preds = outputs.argmax(dim=1)
        bs = labels.size(0)
        running_loss += loss.item() * bs
        correct += (preds == labels).sum().item()
        seen += bs
    return running_loss / seen, correct / seen


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    y_true = []
    y_pred = []
    seen = 0
    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        preds = outputs.argmax(dim=1)
        bs = labels.size(0)
        running_loss += loss.item() * bs
        seen += bs
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
    result = metrics_from_predictions(y_true, y_pred)
    result["loss"] = float(running_loss / seen)
    return result, np.array(y_true), np.array(y_pred)


def run_trial(hp, train_idx, val_idx):
    train_dataset, _ = normalized_imagefolder(TRAIN_DIR, hp["image_size"], augment=hp["augment"])
    val_dataset, _ = normalized_imagefolder(TRAIN_DIR, hp["image_size"], augment=False)
    train_loader = DataLoader(Subset(train_dataset, train_idx), batch_size=hp["batch_size"], shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
    val_loader = DataLoader(Subset(val_dataset, val_idx), batch_size=hp["batch_size"], shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

    model = build_model(hp).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(model, hp)

    history = []
    for epoch in range(1, hp["trial_epochs"] + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_metrics, _, _ = evaluate(model, val_loader, criterion)
        history.append({"epoch": epoch, "train_loss": float(train_loss), "train_accuracy": float(train_acc), **val_metrics})
        print(
            f"trial epoch {epoch}/{hp['trial_epochs']} - "
            f"train_acc={train_acc:.4f} "
            f"val_fake_f1={val_metrics['f1_fake_positive']:.4f} "
            f"val_weighted_f1={val_metrics['weighted_f1']:.4f}"
        )

    best_epoch = max(history, key=lambda row: row["weighted_f1"])
    return {"hyperparameters": copy.deepcopy(hp), "best_validation": best_epoch, "history": history}


def run_hyperparameter_tuning():
    base_image_size = HYPERPARAMETER_GRID[0]["image_size"]
    train_idx, val_idx = make_tuning_splits(base_image_size, train_per_class=TUNING_TRAIN_PER_CLASS, val_per_class=TUNING_VAL_PER_CLASS)
    results = []
    for trial_number, hp in enumerate(HYPERPARAMETER_GRID, start=1):
        print("=" * 80)
        print(f"Trial {trial_number}/{len(HYPERPARAMETER_GRID)}")
        print(json.dumps(hp, indent=2))
        result = run_trial(hp, train_idx, val_idx)
        results.append(result)
        with open(TUNING_RESULTS_PATH, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=4)

    best = max(results, key=lambda row: row["best_validation"]["weighted_f1"])
    with open(BEST_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(best, f, indent=4)
    print("Best tuned hyperparameters:")
    print(json.dumps(best["hyperparameters"], indent=2))
    print("Best validation metrics:")
    print(json.dumps(best["best_validation"], indent=2))
    return best, results


## Final Training With Best Hyperparameters

After selecting the best validation F1 score, the notebook trains the final model using that best configuration and evaluates it on the official test set.

In [ ]:
def train_final_model(best_hp):
    final_hp = copy.deepcopy(best_hp)
    final_hp["trial_epochs"] = FINAL_EPOCHS
    train_dataset, _ = normalized_imagefolder(TRAIN_DIR, final_hp["image_size"], augment=final_hp["augment"])
    test_dataset, _ = normalized_imagefolder(TEST_DIR, final_hp["image_size"], augment=False)
    train_loader = DataLoader(train_dataset, batch_size=final_hp["batch_size"], shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
    test_loader = DataLoader(test_dataset, batch_size=final_hp["batch_size"], shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

    model = build_model(final_hp).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(model, final_hp)

    for epoch in range(1, FINAL_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        print(f"final epoch {epoch}/{FINAL_EPOCHS} - train_loss={train_loss:.4f} train_acc={train_acc:.4f}")

    test_metrics, y_true, y_pred = evaluate(model, test_loader, criterion)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    report_text = classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=class_names,
        zero_division=0,
    )
    torch.save(model.state_dict(), BEST_MODEL_PATH)
    final_results = {
        "model": MODEL_NAME,
        "best_hyperparameters": final_hp,
        "official_test_metrics": test_metrics,
        "confusion_matrix": cm.tolist(),
        "classification_report": report_dict,
        "checkpoint": str(BEST_MODEL_PATH),
    }
    with open(FINAL_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(final_results, f, indent=4)
    print("Saved best tuned checkpoint to:", BEST_MODEL_PATH)
    print("Official test metrics:")
    print(f"Accuracy: {test_metrics['accuracy']:.4f}")
    print(f"Precision, fake positive: {test_metrics['precision_fake_positive']:.4f}")
    print(f"Recall, fake positive: {test_metrics['recall_fake_positive']:.4f}")
    print(f"F1, fake positive: {test_metrics['f1_fake_positive']:.4f}")
    print(f"Weighted precision: {test_metrics['weighted_precision']:.4f}")
    print(f"Weighted recall: {test_metrics['weighted_recall']:.4f}")
    print(f"Weighted F1: {test_metrics['weighted_f1']:.4f}")
    print(f"Test loss: {test_metrics['loss']:.4f}")
    print("Confusion matrix [[fake->fake, fake->real], [real->fake, real->real]]:")
    print(cm)
    print("Classification report:")
    print(report_text)
    return final_results


best_trial, all_trials = run_hyperparameter_tuning()
final_results = train_final_model(best_trial["hyperparameters"])
